# Phase IV: topological analysis of sample-specific weighted PPI networks using betweenness and eigenvector centrality 

## Part 1: Create Control and Patient PPI Networks with Distance Information

In [ ]:
import pandas as pd
import networkx as nx

In [ ]:
# Create empty graphs for control and patient PPI networks
G_control = nx.Graph()
G_patient = nx.Graph()

In [ ]:
# Upload the control and patient PPI dataframes with distance information
control_PPI_df = pd.read_csv("../data/STRING/control_PPI_dist.tsv", sep="\t")
patient_PPI_df = pd.read_csv("../data/STRING/patient_PPI_dist.tsv", sep="\t")

In [ ]:
# Add edges to the control PPI graph with distance as an attribute
for _, row in control_PPI_df.iterrows():
    G_control.add_edge(row['protein1'], row['protein2'], Distance=row['Distance'])

# Check the number of edges and nodes in the control PPI graph
print(f"Number of nodes in control PPI graph: {G_control.number_of_nodes()}")
print(f"Number of edges in control PPI graph: {G_control.number_of_edges()}")

Number of nodes in control PPI graph: 4430
Number of edges in control PPI graph: 89431


In [ ]:
# Add edges to the patient PPI graph with distance as an attribute
for _, row in patient_PPI_df.iterrows():
    G_patient.add_edge(row['protein1'], row['protein2'], Distance=row['Distance'])

# Check the number of edges and nodes in the patient PPI graph
print(f"Number of nodes in patient PPI graph: {G_patient.number_of_nodes()}")
print(f"Number of edges in patient PPI graph: {G_patient.number_of_edges()}")

Number of nodes in patient PPI graph: 4430
Number of edges in patient PPI graph: 89431


## Part 2: Calculate Betweenness and Eigenvector Centrality for Control and Patient PPI Networks

In [ ]:
# #This took around 40 min to run 
# # Calculate betweenness centrality for control network
# bet_control = nx.betweenness_centrality(G_control, weight='Distance')

# # Calculate betweenness centrality for patient network
# bet_patient = nx.betweenness_centrality(G_patient, weight='Distance')

In [ ]:
# # Save betweenness centrality results to TSV files
# # Control network
# bet_control_df = pd.DataFrame(bet_control.items(), columns=['proteins', 'betweenness_centrality'])
# bet_control_df.to_csv("../data/results/betweenness_control.tsv",sep="\t", index=False)
# # Patient network
# bet_patient_df = pd.DataFrame(bet_patient.items(), columns=['proteins', 'betweenness_centrality'])
# bet_patient_df.to_csv("../data/results/betweenness_patient.tsv",sep="\t", index=False)

In [ ]:
# Convert distance to correlation for both control and patient networks
# According to the documentation, weight is interpreted as the connection strength by Eigenvector centrality,
# so convert distance back to correlation (similarity) for centrality calculations
# distance = 1 - absolute correlation from earlier steps, so correlation = 1 - distance
G_control_copy = G_control.copy()
G_patient_copy = G_patient.copy()

# Update edge attributes to convert distance back to correlation
for u, v, data in G_control_copy.edges(data=True):
    data['correlation'] = 1 - data['Distance']

# Update edge attributes to convert distance back to correlation for patient network
for u, v, data in G_patient_copy.edges(data=True):
    data['correlation'] = 1 - data['Distance']

In [ ]:
# Calculate eigenvector centrality for control network
eigenvector_control = nx.eigenvector_centrality(G_control_copy, weight='correlation')

# Calculate eigenvector centrality for patient network
eigenvector_patient = nx.eigenvector_centrality(G_patient_copy, weight='correlation')

# Phase V: DiCE genes identified by gene prioritization using ensemble ranking based on absolute differences of each centrality measure

### Part 1: Use "distance" as weight in betweenness and "correlation" as weight in eigenvector centrality calculations for Control and Patient PPI networks

In [ ]:
# Convert all metrics to a dataframe 
metrics_df = pd.DataFrame({
    'betweenness_control': bet_control,
    'betweenness_patient': bet_patient,
    'eigenvector_control': eigenvector_control,
    'eigenvector_patient': eigenvector_patient
})

print(metrics_df.shape)
metrics_df.head()

(4430, 4)


,betweenness_control,betweenness_patient,eigenvector_control,eigenvector_patient
ARF5,0.000105,0.000775,0.000292,0.000488
RAB1B,0.000094,0.000408,0.000367,0.000397
RAB6A,0.001769,0.000162,0.001626,0.001217
GNB2,0.001208,0.002059,0.002521,0.002987
MAPK8IP3,0.000034,0.000028,0.000337,0.000172


In [ ]:
# Add absolute difference columns for both metrics
metrics_df['delta_betweenness'] = abs(metrics_df['betweenness_patient'] - metrics_df['betweenness_control'])
metrics_df['delta_eigenvector'] = abs(metrics_df['eigenvector_patient'] - metrics_df['eigenvector_control'])

In [ ]:
# Rank the proteins based on the absolute differences for both metrics
metrics_df['rank_btwn'] = metrics_df['delta_betweenness'].rank(ascending=False)
metrics_df['rank_eig'] = metrics_df['delta_eigenvector'].rank(ascending=False)

In [ ]:
# Normalize ranks to a 0-1 scale (1 = highest importance, 0 = lowest importance)
n = len(metrics_df)
metrics_df['norm_rank_btwn'] = 1 - (metrics_df['rank_btwn'] - 1) / (n - 1)
metrics_df['norm_rank_eig'] = 1 - (metrics_df['rank_eig'] - 1) / (n - 1)

In [ ]:
# Add ensembl score by taking product of normalized ranks
metrics_df['ensembl_score'] = metrics_df['norm_rank_btwn'] * metrics_df['norm_rank_eig']

In [ ]:
# # Save the all metrics and scores to a TSV file before filtering
# metrics_df.to_csv("../data/results/btwn_eig_metrics.tsv", sep="\t", index=True)

In [ ]:
# Get mean values for betweenness and eigenvector centrality in control and patient networks
mean_btwn_ctrl = metrics_df['betweenness_control'].mean()
mean_btwn_pat  = metrics_df['betweenness_patient'].mean()
mean_eig_ctrl  = metrics_df['eigenvector_control'].mean()
mean_eig_pat   = metrics_df['eigenvector_patient'].mean()

# Filter out genes that are below average in both metrics for both control and patient networks
filter = ~(
    (metrics_df['betweenness_control'] < mean_btwn_ctrl) & (metrics_df['betweenness_patient'] < mean_btwn_pat) &
    (metrics_df['eigenvector_control'] < mean_eig_ctrl)  & (metrics_df['eigenvector_patient'] < mean_eig_pat)
)

dice_genes = metrics_df[filter].sort_values('ensembl_score', ascending=False)
print(f"Number of DiCE genes: {len(dice_genes)}")
dice_genes.head()

Number of DiCE genes: 1332


,betweenness_control,betweenness_patient,eigenvector_control,eigenvector_patient,delta_betweenness,delta_eigenvector,rank_btwn,rank_eig,norm_rank_btwn,norm_rank_eig,ensembl_score
ACTB,0.011712,0.256875,0.031004,0.063022,0.245162,0.032018,1.0,23.0,1.000000,0.995033,0.995033
HSPA8,0.004188,0.024955,0.046247,0.079928,0.020767,0.033680,12.0,21.0,0.997516,0.995484,0.993012
RPL19,0.001009,0.014445,0.072468,0.116664,0.013436,0.044196,27.0,10.0,0.994130,0.997968,0.992109
BRD4,0.000477,0.023247,0.011787,0.038285,0.022770,0.026498,10.0,29.0,0.997968,0.993678,0.991659
RPL27A,0.000881,0.023443,0.089205,0.115321,0.022562,0.026116,11.0,32.0,0.997742,0.993001,0.990759


In [ ]:
# # Save the results to a TSV file
# dice_genes.to_csv("../data/results/dice_genes.tsv", sep="\t", index=True)

### Part 2: Use "distance" as weight in betweenness and eigenvector centrality calculations for Control and Patient PPI networks

In [ ]:
# Upload the betweenness from the tsv files
bet_control_df = pd.read_csv("../data/results/betweenness_control.tsv", sep="\t")
bet_control_df.rename(columns= {"betweenness_centrality":"betweenness_controls"}, inplace=True)
bet_patient_df = pd.read_csv("../data/results/betweenness_patient.tsv", sep="\t")
bet_patient_df.rename(columns= {"betweenness_centrality":"betweenness_patients"},inplace=True)

In [ ]:
# Calculate eigenvector centrality whiles using distance as weight 
# Calculate eigenvector centrality for control network
eigenvector_control_dist = nx.eigenvector_centrality(G_control_copy, weight='Distance')

# Calculate eigenvector centrality for patient network
eigenvector_patient_dist = nx.eigenvector_centrality(G_patient_copy, weight='Distance')

In [ ]:
# Convert eigenvector centrality results to dataframes
eigenvector_control_dist_df = pd.DataFrame(eigenvector_control_dist.items(), columns=['proteins', 'eigenvector_cent_controls'])
eigenvector_patient_dist_df = pd.DataFrame(eigenvector_patient_dist.items(), columns=['proteins', 'eigenvector_cent_patients'])

In [ ]:
# Join betweenness df of controls and patients
res_bet = pd.merge(bet_control_df, bet_patient_df, on=['proteins'], how='inner')

# Join eigenvector centrality of controls and patients
res_ein = pd.merge(eigenvector_control_dist_df,eigenvector_patient_dist_df, on=["proteins"], how="inner")

# Finally join betweenness and eigenvector of controls and patients
results_df = pd.merge(res_bet, res_ein, on = ["proteins"], how="inner")
print(results_df.shape)
results_df.head()

(4430, 5)


,proteins,betweenness_controls,betweenness_patients,eigenvector_cent_controls,eigenvector_cent_patients
0,ARF5,0.000105,0.000775,0.000734,0.000529
1,RAB1B,0.000094,0.000408,0.001738,0.001639
2,RAB6A,0.001769,0.000162,0.002734,0.002724
3,GNB2,0.001208,0.002059,0.006848,0.006454
4,MAPK8IP3,0.000034,0.000028,0.001370,0.001594


In [ ]:
# Add absolute difference columns for both metrics
results_df['delta_betweenness'] = abs(results_df['betweenness_patients'] - results_df['betweenness_controls'])
results_df['delta_eigenvector'] = abs(results_df['eigenvector_cent_patients'] - results_df['eigenvector_cent_controls'])

In [ ]:
# Rank the proteins based on the absolute differences for both metrics
results_df['rank_btwn'] = results_df['delta_betweenness'].rank(ascending=False)
results_df['rank_eig'] = results_df['delta_eigenvector'].rank(ascending=False)

In [ ]:
# Normalize ranks to a 0-1 scale (1 = highest importance, 0 = lowest importance)
n_df = len(results_df)
results_df['norm_rank_btwn'] = 1 - (results_df['rank_btwn'] - 1) / (n_df - 1)
results_df['norm_rank_eig'] = 1 - (results_df['rank_eig'] - 1) / (n_df - 1)

In [ ]:
# Add ensembl score by taking product of normalized ranks
results_df['ensembl_score'] = results_df['norm_rank_btwn'] * results_df['norm_rank_eig']

In [ ]:
# # Save the all metrics and scores to a TSV file before filtering
# results_df.to_csv("../data/results/btwn_eig_results_dist.tsv", sep="\t", index=False)

In [70]:
# Get mean values for betweenness and eigenvector centrality in control and patient networks
avg_btwn_ctrl = results_df['betweenness_controls'].mean()
avg_btwn_pat  = results_df['betweenness_patients'].mean()
avg_eig_ctrl  = results_df['eigenvector_cent_controls'].mean()
avg_eig_pat   = results_df['eigenvector_cent_patients'].mean()

# Filter out genes that are below average in both metrics for both control and patient networks
filter2 = ~(
    (results_df['betweenness_controls'] < avg_btwn_ctrl) & (results_df['betweenness_patients'] < avg_btwn_pat) &
    (results_df['eigenvector_cent_controls'] < avg_eig_ctrl)  & (results_df['eigenvector_cent_patients'] < avg_eig_pat)
)

dice_genes_v2 = results_df[filter2].sort_values('ensembl_score', ascending=False)
print(f"Number of DiCE genes: {len(dice_genes_v2)}")
dice_genes_v2.head()

Number of DiCE genes: 1453


,proteins,betweenness_controls,betweenness_patients,eigenvector_cent_controls,eigenvector_cent_patients,delta_betweenness,delta_eigenvector,rank_btwn,rank_eig,norm_rank_btwn,norm_rank_eig,ensembl_score
316,ACTB,0.011712,0.256875,0.071582,0.054273,0.245162,0.017309,1.0,6.0,1.000000,0.998871,0.998871
357,HSPA8,0.004188,0.024955,0.062392,0.048015,0.020767,0.014377,12.0,13.0,0.997516,0.997291,0.994814
1777,BRD4,0.000477,0.023247,0.043125,0.029972,0.022770,0.013153,10.0,20.0,0.997968,0.995710,0.993687
523,RPL19,0.001009,0.014445,0.080614,0.065964,0.013436,0.014650,27.0,11.0,0.994130,0.997742,0.991885
512,RPL27A,0.000881,0.023443,0.075026,0.065080,0.022562,0.009947,11.0,31.0,0.997742,0.993226,0.990984


In [ ]:
# # Save the results to a TSV file
# dice_genes_v2.to_csv("../data/results/dice_genes_v2.tsv", sep="\t", index=False)